In [27]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    api_key="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",
    model="openai/gpt-oss-20b"
)

print("Ready!")

Ready!


In [28]:
# PakShop ka document
document = """
PakShop Pakistan ka sabse bada online store hai.

PRODUCTS:
- Mobile phones: Samsung, iPhone, Xiaomi
- Laptops: Dell, HP, Lenovo
- Clothing: Men, Women, Kids
- Home appliances: AC, Fridge, Washing machine

DELIVERY POLICY:
- Lahore: 1-2 din
- Karachi: 2-3 din  
- Islamabad: 2-3 din
- Other cities: 3-5 din
- Free delivery orders above Rs. 2000

RETURN POLICY:
- 7 din ke andar return
- Original condition zaroori
- Receipt zaroori

PAYMENT:
- Cash on Delivery
- JazzCash
- Easypaisa
- Credit Card

CONTACT:
- Phone: 0300-1234567
- Email: support@pakshop.pk
- Hours: 9am to 9pm
"""

# Document split karo
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(document)
print(f"Total chunks: {len(chunks)}")
print("\nPehla chunk:")
print(chunks[0])

Total chunks: 5

Pehla chunk:
PakShop Pakistan ka sabse bada online store hai.


In [29]:
prompt_template = ChatPromptTemplate.from_template("""
Tum PakShop ka assistant ho. Sirf is information se jawab do :
{context}
Sawaal : {question}
Roman urdu mein jawab do. Agar jawab nahi pata to "Mujhe nahi pata" likho."""
)
parser = StrOutputParser()

def ask_rag(question):
    #Relevant chunks nikalne ke liye simple keyword search
    relevant_chunks = []
    question_lower = question.lower()

    for chunk in chunks: 
        if any(word in chunk.lower() for word in question_lower.split()):
            relevant_chunks.append(chunk)

    #context banao
    context = "\n".join(relevant_chunks) if relevant_chunks else "\n".join(chunks[:2])

    #chain chalao
    chain = prompt_template | llm | parser
    response = chain.invoke({
        "context" : context,
        "question": question
    })

    return response

#Test karo
print("Q: Delivery Kitne din mein hogi?")
print("A:", ask_rag("Delivery kitne din mein hogi?"))

Q: Delivery Kitne din mein hogi?
A: Lahore: 1‑2 din  
Karachi: 2‑3 din  
Islamabad: 2‑3 din  
Dusre shehron: 3‑5 din


In [30]:
questions = [
    "Return policy kya hai?",
    "Kaunse payment methods hain?",
    "Contact number kya hai?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_rag(q)}")
    print("-" * 40)

Q: Return policy kya hai?
A: Return policy: 7 din ke andar return, original condition zaroori, receipt zaroori.
----------------------------------------
Q: Kaunse payment methods hain?
A: Cash on Delivery, JazzCash, Easypaisa, Credit Card.
----------------------------------------
Q: Contact number kya hai?
A: 0300-1234567
----------------------------------------


In [31]:
print("Q: iPhone 15 ki price kya hai?")
print("A:", ask_rag("iPhone 15 ki price kya hai?"))

Q: iPhone 15 ki price kya hai?
A: Mujhe nahi pata.


In [32]:
from  langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser


llm = ChatGroq(
    api_key="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",
    model="openai/gpt-oss-20b"
)

document = """
City Care Hospital & Medical Center Lahore.

EMERGENCY SERVICES:
- 24/7 Emergency and ICU available
- Ambulance Contact: 042-111222333
- Emergency ward treatment for critical cases

OPD DOCTORS & TIMINGS:
- Dr. Ahmed (Cardiologist): Monday to Wednesday (10:00 AM to 2:00 PM)
- Dr. Sara (Dermatologist): Thursday to Saturday (3:00 PM to 7:00 PM)
- Dr. Usman (Child Specialist): Daily (5:00 PM to 8:00 PM)

APPOINTMENT & FEES:
- OPD Consultation Fee: Rs. 2000
- Advance appointment is required
- Payment Methods: Cash, JazzCash, Easypaisa, Credit Card

LOCATION & CONTACT:
- Address: Main Boulevard, Gulberg III, Lahore
- Phone: 042-99887766
- Email: info@citycarehospital.pk
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(document)
print(f"Total chunks: {len(chunks)}")
print("\nPehla chunk:")
print(chunks[0])

Total chunks: 5

Pehla chunk:
City Care Hospital & Medical Center Lahore.

EMERGENCY SERVICES:
- 24/7 Emergency and ICU available
- Ambulance Contact: 042-111222333
- Emergency ward treatment for critical cases


In [33]:
prompt_template = ChatPromptTemplate.from_template("""
    You are a helpful assistant. Here is the context: 
    {context}.
     Please answer the question: {question} 
     roman urdu mein jawaab dp or agr jawab nahi pata to 'Mujhe nahi pata' likho."""
)

def ask_hosiptal(question):
    relevant_chunks = []
    question_lower = question.lower()
    for chunk in chunks:
        if any(word in chunk.lower() for word in question_lower.split()):
            relevant_chunks.append(chunk)

    context = "\n".join(relevant_chunks) if relevant_chunks else "\n".join(chunks[:2])
# strOutputParser ko hum is liye use kar rahe hain taki output ko string mein convert kar sakein
    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({
        "context": context,
        "question": question
    })
    return response

print("Q: Emergency services kab available hain?")
print("A:", ask_hosiptal("Emergency services kab available hain?"))

Q: Emergency services kab available hain?
A: 24/7 (24 ghante, 7 din)


In [34]:
questions = [
    "Dr. Sara ki timing kya hai?",

    "Emergency kab tk khuli rehti hai?",

    "Dentist ki fee kitni hai?"
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_hosiptal(q)}")
    print("-" * 40)

Q: Dr. Sara ki timing kya hai?
A: Dr. Sara ki timing: Thursday se Saturday tak 3:00 PM se 7:00 PM tak available hain.
----------------------------------------
Q: Emergency kab tk khuli rehti hai?
A: 24 ghante khuli rehti hai.
----------------------------------------
Q: Dentist ki fee kitni hai?
A: Dentist ki fee Rs. 2000 hai.
----------------------------------------
